## Classification Evaluation Metrics

Classification metrics compare a model's predictions with the actual class labels. Different metrics reveal different strengths and weaknesses, so a
model should rarely be evaluated using only one metric.



### Confusion Matrix

A confusion matrix divides predictions into four categories:

| Result | General meaning |
|---|---|
| True Positive (TP) | The model predicts positive, and the actual class is positive. |
| True Negative (TN) | The model predicts negative, and the actual class is negative. |
| False Positive (FP) | The model predicts positive, but the actual class is negative. This is also called a Type I error. |
| False Negative (FN) | The model predicts negative, but the actual class is positive. This is also called a Type II error. |

A confusion matrix is useful because it shows the exact types of correct and incorrect predictions hidden inside summary metrics.

### Accuracy

**Meaning:** The proportion of all predictions that are correct.

**Formula:**

Accuracy = (TP + TN) / (TP + TN + FP + FN)

**Useful when:**

- The classes are reasonably balanced.
- False positives and false negatives have similar costs.

**Limitation:** Accuracy can be misleading on imbalanced datasets. A model may achieve high accuracy simply by predicting the majority class every
time.

### Precision

**Meaning:** Of all cases predicted as positive, the proportion that are actually positive.

**Formula:**

Precision = TP / (TP + FP)

**Useful when:**

- False positives are costly.
- A positive prediction should be trustworthy.
- Examples include spam filtering, fraud investigations, and recommending expensive interventions.

**Interpretation:** High precision means that when the model predicts positive, it is usually correct.

**Limitation:** A model can obtain high precision by making very few positive predictions, potentially missing many actual positives.

### Recall

**Meaning:** Of all actual positive cases, the proportion the model correctly identifies.

Recall is also called **sensitivity** or the **true-positive rate**.

**Formula:**

Recall = TP / (TP + FN)

**Useful when:**

- False negatives are costly.
- Missing a positive case is particularly harmful.
- Examples include disease screening, safety detection, and detecting fraudulent transactions.

**Interpretation:** High recall means that the model finds most of the actual positive cases.

**Limitation:** Increasing recall can produce more false positives and reduce precision.

### F1 Score

**Meaning:** The harmonic mean of precision and recall. It summarizes their balance in one number.

**Formula:**

F1 = 2 × (Precision × Recall) / (Precision + Recall)

**Useful when:**

- The positive class is important.
- The classes are imbalanced.
- Both false positives and false negatives matter.
- A single metric is needed to balance precision and recall.

**Limitation:** F1 ignores true negatives. It also assumes precision and recall are equally important, which may not match the real cost of errors.

### ROC Curve

The Receiver Operating Characteristic (ROC) curve shows model performance across many possible classification thresholds.

Its axes are:

- **True-positive rate:** the proportion of actual positives correctly identified. This is the samestanden as recall.
- **False-positive rate:** the proportion of actual negatives incorrectly predicted as positive.

False-positive rate = FP / (FP + TN)

Eachzahlungen point on the curve represents anerie/request different decision threshold. Lowering the threshold generally increases both the true-
positive rate and the false-positive rate.

**Interpretation:**

- A curve near the upper-left corner indicates better discrimination.
- A diagonal curve indicates performance melt similar toCHF random gist stays consistent guessing.
- A perfect classifier passes through the point (0, 1).

### ROC-AUC

ROC-AUC is the area under the ROC curve. It measures how well the model ranks positive cases above negative cases across all classification
thresholds.

A useful probabilistic interpretation is:

> ROC-AUC is the probability that a randomly selected positive case receives a higher model score than a randomly selected negative case.

General interpretation:

| ROC-AUC | Meaning |
|---|---|
| 1.0 | Perfect class ranking |
| 0.5 | Similar to random ranking |
| Below 0.5 | The ranking is worse than random, possibly because predictions or labels are reversed |

**Useful when:**

- Comparing the ranking ability of classifiers.
- The final decision threshold has not yet been chosen.
- Performance across multiple thresholds matters.

**Limitations:**

- ROC-AUC does not measure accuracy.
- It does not describe performance at one specific threshold.
- It may appear optimistic when the positive class is extremely rare.
- A precision-recall curve may be more informative for highly imbalanced datasets.

## Metric Selection Guide

| Situation | Metric to emphasize |
|---|---|
| Classes are balanced and error costs are similar | Accuracy |
| False positives are especially costly | Precision |
| False negatives are especially costly | Recall |
| Precision and recall both matter | F1 score |
| Exact error counts and types are needed | Confusion matrix |
| Ranking quality across thresholds matters | ROC-AUC and ROC curve |
| The positive class is extremely rare | Precision-recall curve and PR-AUC |

## Preprocessing and Modeling Components

These components create a workflow that converts raw tabular data into a numerical format and then trains a classification model.

### Pipeline

`Pipeline` connects multiple operations and runs them in a fixed sequence.

For example:

Raw data → fill missing values → scale or encode values → train model

During `fit()`, each step learns what it needs from the training data and passes its transformed output to the next step.

During `predict()`, the pipeline applies the transformations learned during training before making predictions.

**Useful because:**

- It keeps preprocessing and modeling together.
- It ensures transformations occur in the correct order.
- It reduces the risk of data leakage.
- It makes the entire workflow easier to reuse.
- It works with tools such as cross-validation and parameter tuning.

### ColumnTransformer

`ColumnTransformer` applies different transformations to different groups of columns.

For example:

- Numerical columns can receive median imputation and scaling.
- Categorical columns can receive mode imputation and one-hot encoding.

After processing each group separately, it combines the results into one transformed feature matrix.

**Useful because:**

- Tabular datasets often contain several data types.
- Numerical and categorical values require different preprocessing.
- It preserves all preprocessing inside one fitted workflow.

The names such as `"numeric"` and `"categorical"` are labels used to identify the transformations. They do not change the calculations.

### SimpleImputer

`SimpleImputer` fills missing values using a rule learned from the training data.

#### Median strategy

`SimpleImputer(strategy="median")` calculates the median of each selected numerical column and replaces missing values in that column with its median.

**Useful when:**

- A numerical column contains missing values.
- Extreme values might make the mean unrepresentative.
- A simple and reproducible missing-value policy is appropriate.

#### Most-frequent strategy

`SimpleImputer(strategy="most_frequent")` finds the most common value in each selected column and uses it to replace missing values.

**Useful when:**

- A categorical column contains missing values.
- The model or next transformation cannot accept missing values directly.
- Replacing missing values with the mode is a reasonable policy.

**Important limitation:** Imputation does not prove that the missing value truly belongs to the replacement category. It is a modeling decision, not a
factual conclusion about the missing observation.

### StandardScaler

`StandardScaler` standardizes numerical columns using the training mean and standard deviation.

Standardized value = (original value − training mean) / training standard deviation

After standardization:

- Values near the mean are close to 0.
- Positive values are above the mean.
- Negative values are below the mean.
- A value's magnitude describes its distance from the mean in standard-deviation units.

**Useful because:**

- Numerical features may have very different ranges.
- Many linear and distance-based models train more reliably with scaled features.
- A large measurement scale should not automatically give a feature greater numerical influence.

**Important:** The scaler must learn its mean and standard deviation from the training data only. The same learned values are then applied to
validation and test data.

### OneHotEncoder

`OneHotEncoder` converts categorical values into numerical indicator columns.

For example, a feature containing:

job = management, technician, services

could become:

| job_management | job_technician | job_services |
|---:|---:|---:|
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |

**Useful because:**

- Most machine-learning models cannot directly process text categories.
- It represents categories without assigning an artificial numerical ranking.
- Each learned category receives its own indicator column.

`handle_unknown="ignore"` means that prediction will not fail when new data contains a category that did not appear during training. The indicator
columns for that feature will be set to zero for the unknown category.

**Important:** `handle_unknown="ignore"` handles unseen categories, not missing values. Missing values are handled separately by `SimpleImputer`.

### LogisticRegression

`LogisticRegression` is a classification algorithm despite containing the word "regression."

It learns coefficients that connect input features to the probability of a class. In binary classification, it generally produces:

- A probability for the negative class.
- A probability for the positive class.
- A final class prediction based on a decision threshold.

**Useful because:**

- It is a simple and widely used classification baseline.
- It produces class probabilities.
- It trains relatively quickly.
- Its coefficients can help describe associations between features and predictions.
- It works well with standardized numerical data and one-hot-encoded categorical data.

`max_iter=1000` allows the training algorithm up to 1,000 iterations to find stable coefficients. It may finish earlier if it converges.

Increasing `max_iter` gives the algorithm more opportunities to converge, but it does not automatically improve model quality.

## What Happens During Training

When the complete pipeline runs:

`model.fit(X_train, y_train)`

it performs the following operations:

1. Selects the numerical and categorical columns.
2. Learns numerical imputation values from `X_train`.
3. Learns numerical means and standard deviations from `X_train`.
4. Learns categorical imputation values from `X_train`.
5. Learns the categories present in `X_train`.
6. Transforms `X_train` into a numerical feature matrix.
7. Learns the logistic-regression coefficients from the transformed training data and `y_train`.

## What Happens During Prediction

When the fitted pipeline runs:

`model.predict(X_test)`

it performs the following operations:

1. Selects the same feature groups from `X_test`.
2. Fills missing values using the values learned from `X_train`.
3. Scales numerical values using the mean and standard deviation learned from `X_train`.
4. One-hot encodes categories using the why? using the category structure learned from `X_train`.
5. Passes the transformed data to the trained classifier.
6. Returns one predicted class for each row.

The pipeline does not relearn its preprocessing rules from `X_test`.

## Class Predictions and Probabilities

`model.predict(X_test)` returns final class predictions such as `"no"` or `"yes"`.

`model.predict_proba(X_test)` returns one probability for each possible class.

For binary classification, its output has two columns:

| Probability of class 0 | Probability of class 1 |
|---:|---:|
| 0.85 | 0.15 |
| 0.30 | 0.70 |

The expression `[:, 1]` selects every row from the second probability column.

A safer approach is to identify the desired class explicitly:

`positive_index = list(model.classes_).index("yes")`

`y_probability = model.predict_proba(X_test)[:, positive_index]`

This avoids assuming that the positive class is always located in the second column.

## Component Summary

| Component | General purpose |
|---|---|
| `Pipeline` | Connects transformations and a model in a fixed order |
| `ColumnTransformer` | Applies different processing to different columns |
| `SimpleImputer` | Replaces missing values using a learned rule |
| `StandardScaler` | Standardizes numerical features |
| `OneHotEncoder` | Converts categorical values into numeric indicators |
| `LogisticRegression` | Learns probabilities and class predictions |